# 🚀 DocStrange GPU Mode - Google Colab

Questo notebook ti permette di utilizzare DocStrange con GPU **gratuita** su Google Colab.

## Prima di iniziare:
1. **Attiva la GPU**: `Runtime` → `Change runtime type` → `Hardware accelerator` → **GPU**
2. Esegui le celle in ordine

---

## 1️⃣ Verifica GPU Disponibile

In [ ]:
# Verifica che la GPU sia attiva
!nvidia-smi

import torch
print(f"\n✅ CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ GPU non disponibile! Vai su Runtime > Change runtime type > GPU")

## 2️⃣ Installazione DocStrange

In [ ]:
# Fix compatibilità NumPy + Installa DocStrange DA GITHUB (include modelli GPU)
print("📦 Installazione DocStrange con modelli GPU...\n")

# 1. Downgrade NumPy per compatibilità con docstrange
!pip install -q numpy==1.26.4

# 2. Installa PyPDF2 per lo splitting di PDF grandi
!pip install -q PyPDF2

# 3. Installa DocStrange DA GITHUB (include script download modelli per GPU mode!)
!pip install -q git+https://github.com/FlorindoPalladino/nanonets.ocr-2.git

# 4. Installa dipendenze di sistema per GPU mode
!apt-get install -qq poppler-utils tesseract-ocr

print("\n" + "="*80)
print("✅ Installazione completata!")
print("="*80)
print("\n⚠️  IMPORTANTE: Riavvia la sessione una volta:")
print("   1. Runtime → Riavvia sessione")
print("   2. Ri-esegui celle: 1 → 2 → 3 → GRANDI VOLUMI")
print("   3. I modelli ML verranno scaricati al primo utilizzo (~2-3 GB)")
print("\n💡 Il download modelli avviene automaticamente alla prima esecuzione!")

## 3️⃣ Upload dei Documenti

Carica i tuoi documenti (PDF, immagini, Word, Excel, etc.)

In [ ]:
from google.colab import files
import os

print("📤 Carica i tuoi documenti (PDF, immagini, DOCX, XLSX, PPTX, etc.)")
uploaded = files.upload()

# Salva i nomi dei file caricati
uploaded_files = list(uploaded.keys())
print(f"\n✅ File caricati: {uploaded_files}")

## 4️⃣ Test GPU Mode - Singolo Documento

In [ ]:
from docstrange import DocumentExtractor
import time

# Inizializza con GPU mode
extractor = DocumentExtractor(gpu=True)

# Processa il primo file caricato
if uploaded_files:
    file_path = uploaded_files[0]
    print(f"🔄 Processando: {file_path}...\n")
    
    start_time = time.time()
    result = extractor.extract(file_path)
    elapsed = time.time() - start_time
    
    print(f"\n⚡ Tempo di processamento: {elapsed:.2f}s")
    print(f"📊 Pagine processate: {len(result.pages) if hasattr(result, 'pages') else 'N/A'}")
    print(f"\n" + "="*80)
    print("📝 Risultato (Markdown):")
    print("="*80)
    print(result.to_markdown()[:2000] + "..." if len(result.to_markdown()) > 2000 else result.to_markdown())
else:
    print("⚠️ Nessun file caricato!")

## 5️⃣ Estrazione Dati Strutturati con JSON Schema

In [ ]:
import json

# Esempio: Estrai dati strutturati da una fattura o documento
if uploaded_files:
    # Definisci lo schema JSON per l'estrazione
    schema = {
        "document_type": "string",
        "date": "string",
        "total_amount": "number",
        "items": [
            {
                "description": "string",
                "quantity": "number",
                "price": "number"
            }
        ]
    }
    
    # Estrai dati strutturati
    structured_data = result.extract_data(json_schema=schema)
    
    print("📋 Dati Strutturati Estratti:")
    print("="*80)
    print(json.dumps(structured_data, indent=2, ensure_ascii=False))
else:
    print("⚠️ Carica prima un documento nella cella precedente!")

## 6️⃣ Estrazione Campi Specifici

In [ ]:
# Estrai solo campi specifici
if uploaded_files:
    fields = result.extract_data(
        specified_fields=[
            "invoice_number",
            "date",
            "total_amount",
            "vendor_name",
            "customer_name"
        ]
    )
    
    print("🎯 Campi Specifici Estratti:")
    print("="*80)
    print(json.dumps(fields, indent=2, ensure_ascii=False))
else:
    print("⚠️ Carica prima un documento!")

## 7️⃣ Esportazione in Vari Formati

In [ ]:
from google.colab import files as colab_files

if uploaded_files and result:
    base_name = os.path.splitext(uploaded_files[0])[0]
    
    # Salva in vari formati
    formats = {
        'markdown': result.to_markdown(),
        'text': result.to_text(),
        'html': result.to_html(),
        'json': json.dumps(result.to_dict(), indent=2, ensure_ascii=False)
    }
    
    for fmt, content in formats.items():
        output_file = f"{base_name}.{fmt}"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"✅ Salvato: {output_file}")
    
    # Download tutti i file
    print("\n📥 Download dei file...")
    for fmt in formats.keys():
        colab_files.download(f"{base_name}.{fmt}")
    
    print("\n✅ Download completati!")
else:
    print("⚠️ Carica e processa prima un documento!")

## 8️⃣ Batch Processing - Multipli Documenti

In [ ]:
import time

if len(uploaded_files) > 1:
    print(f"🔄 Processando {len(uploaded_files)} documenti...\n")
    
    results = []
    total_start = time.time()
    
    for i, file_path in enumerate(uploaded_files, 1):
        print(f"[{i}/{len(uploaded_files)}] Processando: {file_path}")
        start = time.time()
        
        try:
            result = extractor.extract(file_path)
            elapsed = time.time() - start
            results.append({
                'file': file_path,
                'success': True,
                'time': elapsed,
                'result': result
            })
            print(f"  ✅ Completato in {elapsed:.2f}s\n")
        except Exception as e:
            print(f"  ❌ Errore: {e}\n")
            results.append({
                'file': file_path,
                'success': False,
                'error': str(e)
            })
    
    total_elapsed = time.time() - total_start
    successful = sum(1 for r in results if r['success'])
    
    print("="*80)
    print(f"📊 Riepilogo Batch Processing")
    print("="*80)
    print(f"Documenti processati: {successful}/{len(uploaded_files)}")
    print(f"Tempo totale: {total_elapsed:.2f}s")
    print(f"Tempo medio per documento: {total_elapsed/len(uploaded_files):.2f}s")
    
elif len(uploaded_files) == 1:
    print("ℹ️ Carica più documenti per testare il batch processing")
else:
    print("⚠️ Carica dei documenti prima!")

## 🚀 GRANDI VOLUMI - Processing 2000+ Pagine

**Ottimizzato per gestire grandi volumi con gestione intelligente della memoria**

Questa sezione supporta:
- 📄 PDF singoli molto grandi (1000+ pagine) - splitting automatico
- 📚 Batch di molti documenti (totale 2000+ pagine)
- 💾 Gestione ottimizzata della memoria GPU
- 💿 Auto-save progressivo dei risultati
- 📊 Progress tracking dettagliato

In [ ]:
from docstrange import DocumentExtractor
import gc
import torch
import time
from pathlib import Path
from PyPDF2 import PdfReader, PdfWriter
import json

def split_large_pdf(pdf_path, chunk_size=100):
    """Split a large PDF into smaller chunks"""
    try:
        reader = PdfReader(pdf_path)
        total_pages = len(reader.pages)
        
        if total_pages <= chunk_size:
            return [pdf_path]  # No need to split
        
        print(f"📄 PDF con {total_pages} pagine - Splitting in chunk da {chunk_size}...")
        
        chunks = []
        for i in range(0, total_pages, chunk_size):
            writer = PdfWriter()
            end_page = min(i + chunk_size, total_pages)
            
            for page_num in range(i, end_page):
                writer.add_page(reader.pages[page_num])
            
            chunk_file = f"chunk_{i//chunk_size}_{Path(pdf_path).stem}.pdf"
            with open(chunk_file, "wb") as f:
                writer.write(f)
            chunks.append(chunk_file)
            print(f"  ✅ Chunk {i//chunk_size + 1}: pagine {i+1}-{end_page}")
        
        return chunks
    except Exception as e:
        print(f"⚠️ Impossibile splittare {pdf_path}: {e}")
        return [pdf_path]

def cleanup_memory():
    """Free up GPU and RAM memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def process_large_volume(files_list, batch_size=50, chunk_size=100, output_dir="results"):
    """
    Process large volume of documents with memory management
    
    Args:
        files_list: List of file paths to process
        batch_size: Number of files to process before memory cleanup
        chunk_size: Pages per chunk for large PDFs
        output_dir: Directory to save results
    """
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    # Initialize extractor
    extractor = DocumentExtractor(gpu=True)
    
    # Prepare files (split large PDFs)
    print("🔍 Analisi e preparazione documenti...\n")
    all_files_to_process = []
    original_file_mapping = {}  # Map chunks to original files
    
    for file_path in files_list:
        if file_path.lower().endswith('.pdf'):
            chunks = split_large_pdf(file_path, chunk_size)
            all_files_to_process.extend(chunks)
            for chunk in chunks:
                original_file_mapping[chunk] = file_path
        else:
            all_files_to_process.append(file_path)
            original_file_mapping[file_path] = file_path
    
    total_files = len(all_files_to_process)
    print(f"\n📊 Totale file da processare: {total_files}")
    print(f"💾 Batch size: {batch_size} (cleanup memoria ogni {batch_size} file)")
    print("="*80 + "\n")
    
    # Process files
    results = []
    start_time = time.time()
    
    for i, file_path in enumerate(all_files_to_process, 1):
        print(f"[{i}/{total_files}] 🔄 {file_path}")
        
        try:
            # Process document
            file_start = time.time()
            result = extractor.extract(file_path)
            elapsed = time.time() - file_start
            
            # Save result immediately
            original_file = original_file_mapping[file_path]
            base_name = Path(file_path).stem
            output_base = Path(output_dir) / base_name
            
            # Save in multiple formats
            with open(f"{output_base}.md", 'w', encoding='utf-8') as f:
                f.write(result.to_markdown())
            with open(f"{output_base}.json", 'w', encoding='utf-8') as f:
                json.dump(result.to_dict(), f, indent=2, ensure_ascii=False)
            
            results.append({
                'file': file_path,
                'original_file': original_file,
                'success': True,
                'time': elapsed,
                'pages': len(result.pages) if hasattr(result, 'pages') else 'N/A'
            })
            
            print(f"  ✅ {elapsed:.1f}s - Salvato in {output_dir}/")
            
        except Exception as e:
            print(f"  ❌ Errore: {e}")
            results.append({
                'file': file_path,
                'original_file': original_file_mapping[file_path],
                'success': False,
                'error': str(e)
            })
        
        # Memory cleanup every batch_size files
        if i % batch_size == 0:
            print(f"\n🧹 Pulizia memoria GPU/RAM... ({i}/{total_files})")
            cleanup_memory()
            
            # Show progress stats
            elapsed_total = time.time() - start_time
            avg_time = elapsed_total / i
            remaining = total_files - i
            eta = avg_time * remaining
            print(f"📊 Progresso: {i/total_files*100:.1f}% | ETA: {eta/60:.1f} min\n")
    
    # Final statistics
    total_elapsed = time.time() - start_time
    successful = sum(1 for r in results if r['success'])
    failed = total_files - successful
    
    print("\n" + "="*80)
    print("🎉 PROCESSAMENTO COMPLETATO!")
    print("="*80)
    print(f"✅ Successi: {successful}/{total_files}")
    print(f"❌ Falliti: {failed}/{total_files}")
    print(f"⏱️  Tempo totale: {total_elapsed/60:.1f} minuti")
    print(f"⚡ Tempo medio per file: {total_elapsed/total_files:.1f}s")
    print(f"📁 Risultati salvati in: {output_dir}/")
    print("="*80)
    
    # Save summary report
    summary_path = Path(output_dir) / "processing_summary.json"
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump({
            'total_files': total_files,
            'successful': successful,
            'failed': failed,
            'total_time_seconds': total_elapsed,
            'avg_time_per_file': total_elapsed/total_files,
            'results': results
        }, f, indent=2, ensure_ascii=False)
    
    print(f"\n📝 Report dettagliato salvato: {summary_path}")
    
    return results

# ========== ESEGUI PROCESSING GRANDI VOLUMI ==========

if uploaded_files and len(uploaded_files) > 0:
    print("🚀 MODALITÀ GRANDI VOLUMI ATTIVATA\n")
    print("⚙️ Configurazione:")
    print("  - Batch size: 50 documenti (cleanup memoria ogni 50)")
    print("  - Chunk size: 100 pagine (per PDF grandi)")
    print("  - Auto-save: Abilitato (salvataggio progressivo)\n")
    
    # Process all uploaded files
    results = process_large_volume(
        files_list=uploaded_files,
        batch_size=50,        # Cleanup memoria ogni 50 file
        chunk_size=100,       # Split PDF grandi ogni 100 pagine
        output_dir="large_volume_results"
    )
    
    # Mostra file disponibili per download
    print("\n📦 File pronti per il download:")
    output_files = list(Path("large_volume_results").glob("*"))
    for f in output_files[:10]:  # Mostra primi 10
        print(f"  - {f.name}")
    if len(output_files) > 10:
        print(f"  ... e altri {len(output_files)-10} file")
    
else:
    print("⚠️ Carica prima dei documenti nella cella 3!")
    print("\n💡 Questa sezione è ottimizzata per:")
    print("  • PDF singoli fino a 2000+ pagine")
    print("  • Batch di centinaia di documenti")
    print("  • Processing con gestione memoria automatica")
    print("  • Salvataggio progressivo dei risultati")

## 9️⃣ Confronto: GPU Mode vs Cloud Mode

In [ ]:
if uploaded_files:
    file_path = uploaded_files[0]
    
    print("⚡ Confronto prestazioni GPU vs Cloud...\n")
    
    # Test GPU Mode
    print("🎮 GPU Mode:")
    extractor_gpu = DocumentExtractor(gpu=True)
    start = time.time()
    result_gpu = extractor_gpu.extract(file_path)
    gpu_time = time.time() - start
    print(f"   Tempo: {gpu_time:.2f}s")
    
    # Test Cloud Mode
    print("\n☁️ Cloud Mode:")
    extractor_cloud = DocumentExtractor(gpu=False)
    start = time.time()
    try:
        result_cloud = extractor_cloud.extract(file_path)
        cloud_time = time.time() - start
        print(f"   Tempo: {cloud_time:.2f}s")
        
        print("\n" + "="*80)
        print("📊 Risultati:")
        print("="*80)
        if gpu_time < cloud_time:
            speedup = cloud_time / gpu_time
            print(f"🚀 GPU Mode è {speedup:.1f}x più veloce!")
        else:
            print(f"☁️ Cloud Mode è più veloce in questo caso")
    except Exception as e:
        print(f"   ⚠️ Errore Cloud Mode: {e}")
        print(f"   (Possibile limite rate limit - GPU Mode funziona comunque!)")
else:
    print("⚠️ Carica prima un documento!")

## 🔟 Test con Documento di Esempio

Se non hai documenti, scarica e testa con un PDF di esempio

In [ ]:
# Scarica un PDF di esempio
!wget -q -O sample_invoice.pdf "https://www.africau.edu/images/default/sample.pdf"

print("✅ Documento di esempio scaricato: sample_invoice.pdf")
print("\n🔄 Processando documento di esempio...\n")

extractor = DocumentExtractor(gpu=True)
result = extractor.extract("sample_invoice.pdf")

print("="*80)
print("📝 Risultato:")
print("="*80)
print(result.to_markdown()[:1500] + "...")
print("\n✅ Test completato con successo!")

---

## 📚 Documentazione e Risorse

- **Repository GitHub**: https://github.com/NanoNets/nanonets.ocr-2
- **API Key** (per Cloud Mode): https://app.nanonets.com/#/keys
- **Supporto**: support@nanonets.com

## 💡 Tips:

1. **GPU gratuita su Colab ha limiti di tempo** (~12 ore/giorno)
2. **Per grandi volumi**: Considera Cloud Mode autenticato (10k docs/mese gratis)
3. **Dati sensibili**: GPU Mode è ideale (tutto locale, nessun upload)
4. **Performance**: GPU Mode è più veloce per batch processing

---

### 🎉 Buon divertimento con DocStrange!